## Baixei tudo local e vou carregar num volume.

Comprimi todos os verbetes em um zip em /Volumes/dhbb/bronze/zona-bruta/text.zip

In [0]:
!tar vxfz /Volumes/public-data/cpdoc/dhbb/dhbb.tar.gz -C /Volumes/public-data/cpdoc/dhbb/zona-bruta

In [0]:
! ls -1 /Volumes/public-data/cpdoc/dhbb/zona-bruta | wc -l
#depois que deu tudo certo, pode apagar o zip
# tem que achar 7864 verbetes
# 7863


In [0]:
import os
import re
import glob
import yaml

def extract_metadata_from_yaml(content):
    metadata = {
        "person_name": "Nome não identificado",
        "natureza": None,
        "sexo": None,
        "cargos": []
    }
    try:
        yaml_match = re.search(r'^---\s*\n(.*?)\n---', content, re.DOTALL | re.MULTILINE)
        if yaml_match:
            yaml_content = yaml_match.group(1)
            title_match = re.search(r'title:\s*(.+?)(?:\n|$)', yaml_content)
            if title_match:
                metadata["person_name"] = title_match.group(1).strip().strip('"').strip("'")
            natureza_match = re.search(r'natureza:\s*(.+?)(?:\n|$)', yaml_content)
            if natureza_match:
                metadata["natureza"] = natureza_match.group(1).strip()
            sexo_match = re.search(r'sexo:\s*(.+?)(?:\n|$)', yaml_content)
            if sexo_match:
                metadata["sexo"] = sexo_match.group(1).strip()
            cargos_section = re.search(r'cargos:\s*\n((?:\s+-\s+.+\n?)+)', yaml_content)
            if cargos_section:
                cargos_text = cargos_section.group(1)
                metadata["cargos"] = [
                    line.strip().lstrip('- ').strip() 
                    for line in cargos_text.split('\n') 
                    if line.strip().startswith('-')
                ]
    except Exception as e:
        print(f"  ⚠️ Erro ao extrair metadados YAML: {str(e)}")
    return metadata

file_contents = []
for file_path in sorted(glob.glob("/Volumes/public-data/cpdoc/dhbb/zona-bruta/*.text")):
    try:
        file_name = os.path.basename(file_path)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            yaml_metadata = extract_metadata_from_yaml(content)
            file_id = file_name.split('.')[0]
            file_contents.append((
                file_id,
                file_name,
                content,
                yaml_metadata["person_name"],
                yaml_metadata["natureza"],
                yaml_metadata["sexo"],
                ", ".join(yaml_metadata["cargos"]) if yaml_metadata["cargos"] else None
            ))
    except Exception as e:
        print(f" ✗ Erro ao carregar {file_path}: {str(e)}")

df = spark.createDataFrame(
    file_contents,
    schema=['id', 'file_name', 'content', 'person_name', 'natureza', 'sexo', 'cargos']
)

In [0]:
df.write.mode("overwrite").saveAsTable("`public-data`.cpdoc.dhbb_bronze")

In [0]:
%sql
select count(*) from `public-data`.cpdoc.dhbb_bronze;
--# deveriam ter 
--# julio.chaves@ERJ0697 MINGW64 /c/Backup/MyGit/dhbb/text (master)
--# $ ls -l |wc -l
--# 7864


In [0]:
%sql
select * from `public-data`.cpdoc.dhbb_bronze where content like '%Marta Suplicy%'